<a href="https://colab.research.google.com/github/Preetish-ko/Preetish-ko/blob/main/Assignments/Assignment_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment 2



## Question 1

A nuclear fuel pellet is a cylinder, 1.5 cm in lenth and 1 cm in diameter. Assume the surface temperature is 300 C everywhere. Given temperature probe data below, determine the radial temperature profile in the middle of a nuclear fuel pellet (i.e.: T(r, z = 0.75)) using radial basis functions.


In [5]:
import numpy as np

# 20 data points presented in columns: | x | y | z | T |, measurements in cm

data = np.array([
    [5.1690e-02, 2.3766e-01, 6.7059e-01, 5.2645e+02],
    [1.1353e-01, 9.4708e-02, 5.3856e-01, 5.5201e+02],
    [1.6676e-01, 1.4358e-01, 4.6936e-01, 5.0802e+02],
    [1.3610e-01, 3.7207e-02, 2.1694e-01, 4.3663e+02],
    [8.9225e-02, 3.7293e-01, 1.1270e+00, 3.9234e+02],
    [1.9001e-01, 3.7240e-01, 8.4774e-01, 3.8872e+02],
    [5.4849e-02, 3.5425e-01, 5.7478e-01, 4.3784e+02],
    [1.7001e-01, 2.0241e-01, 1.2960e+00, 4.0159e+02],
    [2.0606e-01, 3.1594e-01, 6.4077e-01, 4.2652e+02],
    [2.5382e-01, 2.5859e-01, 4.8610e-01, 4.2481e+02],
    [5.6038e-02, 8.2231e-02, 4.2029e-01, 5.3244e+02],
    [3.1242e-01, 8.0489e-02, 1.1530e+00, 4.2453e+02],
    [6.0186e-02, 4.4891e-01, 3.9941e-01, 3.4207e+02],
    [1.5070e-01, 3.4794e-01, 1.5595e-01, 3.4750e+02],
    [1.8215e-01, 3.4388e-01, 1.0478e+00, 3.9963e+02],
    [1.1633e-01, 4.1011e-01, 5.5001e-01, 3.7611e+02],
    [1.2377e-01, 3.3703e-01, 3.7672e-02, 3.1423e+02],
    [4.6378e-02, 3.3653e-01, 1.4434e+00, 3.2345e+02],
    [2.9063e-02, 3.2584e-02, 2.3977e-01, 4.5993e+02],
    [2.1162e-02, 3.8590e-01, 2.5905e-01, 3.6901e+02]
])

Consider what you know about this system. What extra information do you have in terms of

### a) type(s) of symmetry?

#### 1. Axisymmetry
Nothing depends on the angle theta, it is all the same if we take the cylinder as a 2d face where the centre is the axis of rotation. So we can summarize this as a 2d problem where r = √(x²+y²).

#### 2. Mirror symmetry about the midplane

Assuming the midplane is dividing the shape along the z-axis, the surface area from the top and the bottom will be exaclty the same due to the constant surface temperature at every point being 300.

#### 3. Regularity at the axis.
T is even in r, so ∂T/∂r = 0 at r = 0. This means the temperature lines are smooth along this axis.

### b) Boundary conditions?

T = 300 °C on the lateral surface (r = 0.5, all z).


T = 300 °C on both end caps (z = 0 and z = 1.5).


∂T/∂r = 0 at r = 0 and ∂T/∂z = 0 at z = 0.75 (from the symmetries).

## c) Plot the best guess of the radial temperature profile

Note: RBFs will fail with a linear solver error if two data points exactly overlap.

1. First we will need to convert the 3D points into 2D points using symmetries. This reduces the calculations that need to be made by the program when created the graph. it also makes a more easier to understand system for the program so it can calculate it faster.

2. We add the known boundary data as extra points. Since we know T is 300 on the surfaces, we can place points at those location.

3. We can enforce a flat profile at the axis. this is so the mirrored copies make the data symmetric about r=0 and the function would have a slop of zero.

4. Remove duplicated points. Since steps 2 and 3 create points that can overlap with eachother, this would cause an error witht he RBF solver. Removing the duplicated points would thus remove the error.

5. Fit the RBF. Since the RBF is a sum of terms that is centered on each point, each term will depend on the distance on the point.

In [6]:
import numpy as np
from scipy.interpolate import RBFInterpolator

x, y, z, T = data.T
r  = np.hypot(x, y)
zf = np.minimum(z, 1.5 - z)             # folded about the midplane

def loocv(X, kern='thin_plate_spline', sm=1e-3):
    """RMS leave-one-out prediction error for a given choice of coordinates."""
    err = []
    for i in range(len(T)):
        m = np.arange(len(T)) != i
        f = RBFInterpolator(X[m], T[m], kernel=kern, smoothing=sm)
        err.append(f(X[i:i+1])[0] - T[i])
    return np.sqrt(np.mean(np.square(err)))

print("3D (x,y,z)    :", loocv(np.c_[x, y, z]))     # no symmetry assumed
print("axisym (r,z)  :", loocv(np.c_[r, z]))        # axisymmetry only
print("axisym+mirror :", loocv(np.c_[r, zf]))       # both symmetries

# Angular dependence: residuals of the axisymmetric fit vs theta
f = RBFInterpolator(np.c_[r, zf], T, kernel='thin_plate_spline', smoothing=1e-3)
res = T - f(np.c_[r, zf])
print(np.corrcoef(res, np.arctan2(y, x))[0, 1])

# Direct mirror test: fit unfolded (r,z), compare T(r,z) with T(r,L-z)
g = RBFInterpolator(np.c_[r, z], T, kernel='thin_plate_spline', smoothing=1e-3)
rr, zz = np.meshgrid(np.linspace(.05, .4, 8), np.linspace(.3, .7, 5))
print(np.abs(g(np.c_[rr.ravel(), zz.ravel()]) - g(np.c_[rr.ravel(), 1.5 - zz.ravel()])).max())

3D (x,y,z)    : 15.713451575812371
axisym (r,z)  : 8.501915972424907
axisym+mirror : 6.304937995727932
-0.15543282661836968
50.556967344489976


# Question 2

You run an experiment and obtain the following data:

| x | y1 | y2 | y3 | y4 | y5 |
|---|---|---|---| --- | --- |
| 0.00 | -29.49 | -2.14 | 15.88 | 22.69 | 28.53 |
| 1.11 | 2.83 | 18.02 | -25.45 | -32.45 | 7.50 |
| 2.22 | 1.97 | -10.49 | -0.18 | -32.10 | -40.31 |
| 3.33 | -38.09 | -46.16 | -7.87 | -33.97 | -38.39 |
| 4.44 | -3.97 | -32.22 | -33.95 | -11.07 | -32.47 |
| 5.56 | 4.45 | -10.88 | 20.43 | 6.57 | -8.49 |
| 6.67 | 50.22 | 51.29 | 80.02 | 66.15 | 84.90 |
| 7.78 | 164.11 | 190.26 | 160.94 | 182.35 | 163.18 |
| 8.89 | 331.75 | 306.51 | 278.40 | 302.13 | 335.44 |
| 10.00 | 517.06 | 483.20 | 476.73 | 512.16 | 500.64 |



In [7]:
import numpy as np

# Define the table as a list of lists
d = np.array([
    [0.00, -29.49, -2.14, 15.88, 22.69, 28.53],
    [1.11, 2.83, 18.02, -25.45, -32.45, 7.50],
    [2.22, 1.97, -10.49, -0.18, -32.10, -40.31],
    [3.33, -38.09, -46.16, -7.87, -33.97, -38.39],
    [4.44, -3.97, -32.22, -33.95, -11.07, -32.47],
    [5.56, 4.45, -10.88, 20.43, 6.57, -8.49],
    [6.67, 50.22, 51.29, 80.02, 66.15, 84.90],
    [7.78, 164.11, 190.26, 160.94, 182.35, 163.18],
    [8.89, 331.75, 306.51, 278.40, 302.13, 335.44],
    [10.00, 517.06, 483.20, 476.73, 512.16, 500.64]
])

## a) Determine the best cubic polynomial fit to this data with the uncertainty

The model is y = c₀ + c₁x + c₂x² + c₃x³. This is a linear polynomial, so we can use ordinary least squares on the points. The covariance matrix is Σ = s²(AᵀA)⁻¹, where A is the Vandermonde matrix and s² = SSE/(n − 4) is the residual variance. The standard errors are the square roots of its diagonal. The 95% interval is ±t₀.₉₇₅,₄₆ · SE, with t = 2.013.

In [11]:
import numpy as np
from scipy import stats

x = np.repeat(d[:,0], 5)          # 50 x-values
y = d[:,1:].ravel()               # 50 y-values
A = np.vander(x, 4, increasing=True)

c, *_ = np.linalg.lstsq(A, y, rcond=None)

res = y - A @ c                   # residuals
n, p = len(y), A.shape[1]
dof = n - p
sse = res @ res
s   = np.sqrt(sse / dof)

print(n, dof, sse, s)

se   = np.sqrt(np.diag(cov))             # standard error of each coefficient
tval = stats.t.ppf(0.975, dof)           # 95% factor, about 2.013 for 46 dof

# 1) Plain equation
print(f"y = {c[0]:.2f} {c[1]:+.2f} x {c[2]:+.2f} x^2 {c[3]:+.3f} x^3")

# 2) Equation with 95% uncertainties on each coefficient
print(f"y = ({c[0]:.1f} ± {tval*se[0]:.1f})"
      f" + ({c[1]:.1f} ± {tval*se[1]:.1f}) x"
      f" + ({c[2]:.2f} ± {tval*se[2]:.2f}) x^2"
      f" + ({c[3]:.3f} ± {tval*se[3]:.3f}) x^3")
print(f"residual std dev s = {s:.1f}")

50 46 13776.046784484992 17.30546959558124
y = 9.53 -13.09 x -2.59 x^2 +0.881 x^3
y = (9.5 ± 14.1) + (-13.1 ± 13.0) x + (-2.59 ± 3.11) x^2 + (0.881 ± 0.204) x^3
residual std dev s = 17.3


## b) Your manager thinks this should be a quadratic. Which do you think it should be and why?

The cubic would be the better option. The reason its better is because of:

1. The lack of fit test. This uses 5 replicates at each x and gives a model indipdent estimate of the noise. The quadratic would ultimiatly miss the total structure that the noise cannot explain. the cubic however would give a consistent result with the measurement noise. The residual s is also 17.3 which matches the pure noise error of 18.

2. The nestedt f-test. Going from quadratic cutes the SSE from 36369 to 13776.

3. The residual pattern. the quadratucs resudals at x-means run 33, +11, +32, +20, +9, −15, −25, −19, −7, +27. That systematic wave of runs same sign values is a signiture of a missing term, not random noise. The cubic's residuals (−2, 1, 6, −3, 0, −6, −2, 6, 3, −3) scatter around zero with no pattern.

In [12]:
import numpy as np
from scipy import stats

x = np.repeat(d[:,0], 5)
y = d[:,1:].ravel()
n = len(y)

def fit(deg):
    A = np.vander(x, deg+1, increasing=True)
    c = np.linalg.lstsq(A, y, rcond=None)[0]
    r = y - A @ c
    return r @ r, n - (deg+1)            # SSE, degrees of freedom

sse2, dof2 = fit(2)
sse3, dof3 = fit(3)

# Test 1: does adding x^3 significantly reduce the error?
F = ((sse2 - sse3)/(dof2 - dof3)) / (sse3/dof3)
print("F-test p =", stats.f.sf(F, dof2-dof3, dof3))

# Test 2: lack-of-fit vs. the noise measured from the replicates
Y = d[:,1:]
sspe = ((Y - Y.mean(axis=1, keepdims=True))**2).sum()
dfpe = 40
for name, sse, dof in (("quad", sse2, dof2), ("cubic", sse3, dof3)):
    F = ((sse - sspe)/(dof - dfpe)) / (sspe/dfpe)
    print(name, "lack-of-fit p =", stats.f.sf(F, dof-dfpe, dfpe))

# Test 3: AIC / BIC (lower is better; penalises extra parameters)
for name, sse, k in (("quad", sse2, 3), ("cubic", sse3, 4)):
    print(name, "AIC", n*np.log(sse/n) + 2*k, "BIC", n*np.log(sse/n) + k*np.log(n))

# Test 4: cross-validation, leaving out one whole x-value at a time
for deg in (2, 3):
    e = []
    for xv in np.unique(x):
        m = x != xv
        c = np.polyfit(x[m], y[m], deg)
        e += list(y[~m] - np.polyval(c, x[~m]))
    print(deg, "leave-one-x-out RMSE", np.sqrt(np.mean(np.square(e))))

F-test p = 2.945635375041245e-11
quad lack-of-fit p = 2.8192072410634616e-07
cubic lack-of-fit p = 0.8846894298464946
quad AIC 335.47288194593625 BIC 341.2089509622207
cubic AIC 288.93318086599544 BIC 296.581272887708
2 leave-one-x-out RMSE 44.192447674697284
3 leave-one-x-out RMSE 18.477161894214095


####Test 1
asks whether the error drops by more than chance would explain when you add the x³ term. It does, by a huge margin.
####Test 2
compares each model's misfit to the noise measured from repeats. The cubic misses by no more than noise; the quadratic misses by far more.
####Test 3
(AIC/BIC) rewards good fit but charges a penalty per parameter, so the cubic can't win just by having an extra knob. It still wins easily.
####Test 4
asks which model predicts x-values it hasn't seen. Dropping all five readings at one x and predicting them is a strict test; the cubic's error stays at the noise level, while the quadratic's more than doubles.


#overall

the test favours cubic. the F-test shows that there is less of an error. where the quadratic misses the data by more than the replicate the noise allows. The cubic misfit, is fully conistent with noise, with no sign of missing values. the scoring for the AIC and BIC also show that there is a more consistent score with the cubic than the quadratic. Lastly the cubic predicts the x-values about 2.4 times better than the quadratic function.